# Лаборатория 1.1. Как модель видит текст

**Что мы сделаем:** возьмём настоящий токенизатор — тот самый, которым пользуются
модели вроде ChatGPT, — и своими глазами посмотрим, на какие кусочки он режет текст.
Потом посчитаем, во сколько обходится запрос и сколько текста помещается модели «на парту».

**Что понадобится:** ничего. Ни ключей, ни регистраций — токенизатор работает
прямо на этом компьютере.

**Как пользоваться ноутбуком:** ячейки запускаются по порядку, кнопкой ▶ слева
или сочетанием Shift+Enter. Если что-то сломалось — перезапусти с самой первой.

## Шаг 0. Установка библиотеки

`tiktoken` — библиотека от OpenAI, которая умеет резать текст на токены ровно так же,
как это делают их модели. Мы берём её, потому что это **настоящий** инструмент,
а не учебная имитация: те же правила, те же кусочки, те же числа.

Восклицательный знак в начале строки означает «выполни это не как код Python,
а как команду в терминале». `pip` — программа, которая ставит библиотеки.
Флаг `-q` (quiet, «тихо») убирает простыню технических сообщений.

In [ ]:
!pip -q install tiktoken pandas

## Шаг 1. Знакомимся с токенизатором

Токенизатор — это словарь примерно на 100 000 частых кусочков текста плюс правила,
как разложить любой текст на эти кусочки. Каждому кусочку соответствует свой номер:
именно числа, а не буквы, попадают внутрь модели.

`cl100k_base` — название конкретного словаря (его использует, например, GPT-4).
`encode` превращает текст в номера токенов, `decode` — обратно.

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

fraza = "Привет, как дела?"
nomera = tokenizer.encode(fraza)

print("Исходный текст: ", fraza)
print("Номера токенов: ", nomera)
print("Сколько токенов:", len(nomera))

Числа сами по себе ничего не говорят. Расшифруем каждое обратно в текст —
и увидим, на какие именно кусочки распалась фраза.

`decode_single_token_bytes` возвращает кусочек в виде «сырых байтов», поэтому
добавляем `.decode("utf-8", errors="replace")` — превращаем байты в читаемые буквы.
Если попадётся знак `�`, это не ошибка: значит, русская буква не поместилась
в один токен целиком и разрезана посередине. Такое бывает — и это честная картина
того, как модель видит русский текст.

In [ ]:
for nomer in nomera:
    kusochek = tokenizer.decode_single_token_bytes(nomer).decode("utf-8", errors="replace")
    print(f"{nomer:>7} -> {kusochek!r}")

**Останови взгляд на этом выводе.** Три вещи, которые стоит заметить:

1. Пробел — часть токена, а не отдельный символ. Токен ` как` (с пробелом впереди)
   и токен `как` — разные.
2. Знаки препинания обычно отдельные токены.
3. Русские слова разваливаются на несколько кусочков, иногда прямо посреди буквы.

Попробуй поменять `fraza` на что-нибудь своё и запустить обе ячейки заново.

## Шаг 2. Почему русский текст «дороже» английского

Модели учили в основном на английских текстах. Поэтому в словаре токенизатора
частые английские слова лежат целиком, а русские приходится собирать из мелких кусочков.

Сравним пары «одно и то же на двух языках». Считать будем два числа: количество
букв и количество токенов. Заранее не подглядывай в ответ — прикинь сам, во сколько
раз, по-твоему, разойдутся числа.

`pandas` — библиотека для таблиц; берём её, чтобы результат читался как таблица,
а не как каша из принтов.

In [ ]:
import pandas as pd

pary = [
    ("Привет, мир!", "Hello, world!"),
    ("Искусственный интеллект", "Artificial intelligence"),
    ("Сегодня хорошая погода", "The weather is nice today"),
    ("Школьное расписание уроков", "School lesson timetable"),
]

stroki = []
for russkiy, angliyskiy in pary:
    ru_tokenov = len(tokenizer.encode(russkiy))
    en_tokenov = len(tokenizer.encode(angliyskiy))
    stroki.append({
        "русский": russkiy,
        "букв (ру)": len(russkiy),
        "токенов (ру)": ru_tokenov,
        "английский": angliyskiy,
        "токенов (ан)": en_tokenov,
        "во сколько раз дороже": round(ru_tokenov / en_tokenov, 1),
    })

tablica = pd.DataFrame(stroki)
tablica

Последняя колонка — главный вывод лаборатории: **один и тот же смысл на русском
стоит в 2–4 раза больше токенов.** Это не мнение и не оценка «на глазок» —
ты только что посчитал это сам настоящим токенизатором.

Разброс большой: у коротких бытовых фраз множитель меньше, у длинных «умных» слов
больше. Причина простая: «Искусственный интеллект» английский словарь знает целиком
как пару частых слов, а русский вариант собирает из обрывков.

Отсюда следует практическая вещь: если в программе важна цена или скорость,
служебные инструкции модели иногда пишут по-английски, а по-русски оставляют
только то, что видит человек.

## Шаг 3. Сколько стоит запрос

Когда программа обращается к модели через интернет, платит она именно за токены,
причём по разной цене за то, что отправила, и за то, что получила: ответ обычно
в несколько раз дороже.

Цены ниже — порядок величины для средней модели в 2026 году, в долларах за миллион
токенов. Точные числа у каждой модели свои и меняются, поэтому важна не цифра,
а умение прикинуть.

In [ ]:
CENA_ZA_MILLION_VHOD = 0.15    # доллара за миллион отправленных токенов
CENA_ZA_MILLION_VYHOD = 0.60   # доллара за миллион полученных токенов

zapros = "Расскажи, как устроена солнечная система, простыми словами." * 1
otvet_primerno = 400           # столько токенов модель напишет в ответ

vhod = len(tokenizer.encode(zapros))
cena_odnogo = vhod / 1_000_000 * CENA_ZA_MILLION_VHOD + otvet_primerno / 1_000_000 * CENA_ZA_MILLION_VYHOD

print(f"Токенов в запросе: {vhod}")
print(f"Токенов в ответе (оценка): {otvet_primerno}")
print(f"Цена одного запроса: ${cena_odnogo:.6f}")
print(f"Цена 1000 таких запросов: ${cena_odnogo * 1000:.2f}")
print(f"Если школьный бот отвечает 200 раз в день, за месяц: ${cena_odnogo * 200 * 30:.2f}")

Обрати внимание: одна фраза стоит доли цента, и кажется, что считать нечего.
А тысяча запросов — это уже заметные деньги, и именно поэтому инженеры следят
за длиной запросов. Особенно за той частью, которая отправляется **каждый раз**:
длинная инструкция в начале каждого запроса умножается на число запросов.

## Шаг 4. Контекстное окно: сколько модель видит за раз

У модели есть предел: сколько токенов она может держать перед глазами одновременно.
В этот предел входит всё — инструкция, история переписки, найденные документы и
место под ответ.

Посмотрим, что это значит в привычных единицах: страницах текста.

In [ ]:
# Возьмём кусок текста и посчитаем, сколько токенов приходится на одну страницу.
stranica = """Кружок робототехники собирается по вторникам и четвергам в кабинете 204.
Занятие начинается в 15:40 и длится полтора часа. С собой нужно принести тетрадь
и ручку, всё остальное выдаётся на месте. Руководитель кружка — Иванов Пётр Сергеевич.
На первом занятии мы собираем простую тележку на моторчиках и учимся управлять ей
с телефона. Запись у классного руководителя до конца сентября.""" * 3

tokenov_na_stranicu = len(tokenizer.encode(stranica))
print(f"Токенов на страницу текста: примерно {tokenov_na_stranicu}")
print()

for razmer_okna in [4_000, 128_000, 1_000_000]:
    stranic = razmer_okna / tokenov_na_stranicu
    print(f"Окно {razmer_okna:>9,} токенов  ->  примерно {stranic:6.0f} страниц".replace(",", " "))

Даже самое большое окно — это конечное число страниц. И тут есть подвох, о котором
помнят не все: **поместиться — не значит быть замеченным.** Исследователи много раз
показывали, что середину длинного текста модель улавливает хуже, чем начало и конец.

Поэтому в теме 2 мы не будем закидывать в модель всё подряд, а научимся находить
и подставлять только нужные куски.

## Попробуй сам

1. Замени `fraza` в шаге 1 на своё имя и фамилию. Сколько токенов? А если написать
   их латиницей?
2. Добавь в шаг 2 свою пару «русский — английский». Совпал ли множитель с остальными?
3. В шаге 3 поставь `otvet_primerno = 2000`. Насколько выросла цена и почему
   сильнее, чем ты ожидал?
4. Найди слово, которое токенизатор режет на 5 и более кусочков. Подсказка:
   длинные редкие слова, например «сельскохозяйственный».

## Что унести с собой

* Модель видит не буквы и не слова, а **токены** — частые кусочки текста.
* Русский текст даёт в 2–4 раза больше токенов, чем тот же смысл по-английски,
  а значит, дороже и медленнее.
* Платят за токены, причём ответ дороже запроса.
* **Контекстное окно** — предел того, сколько модель видит за раз; середину длинного
  текста она замечает хуже краёв.